In [34]:
from pathlib import Path

import geopandas as gpd


RAIZ_PROYECTO = Path.cwd()
RUTA_BASE = RAIZ_PROYECTO / "base" / "localidades_objetivo_sus.gpkg"
RUTA_SALIDA = RAIZ_PROYECTO / "public" / "mapa_base.geojson"

base = gpd.read_file(RUTA_BASE)

In [27]:
base

,nom_ent,nom_mun,nom_loc,cve_loc,entidad,mun,loc,poblacion_total_2026,tipo,cve_ageb,observacion,tipo_mge,cvegeo,consultorios_faltantes,geometry
0,BAJA CALIFORNIA,ENSENADA,ENSENADA,020010001,02,001,0001,352140.0,Prioridad 1: Ampliamente urbana,None,None,localidad,020010001,NaN,"MULTIPOLYGON (((-116.58237 31.82349, -116.5822..."
1,BAJA CALIFORNIA,MEXICALI,MEXICALI,020020001,02,002,0001,912766.0,Prioridad 1: Ampliamente urbana,None,None,localidad,020020001,NaN,"MULTIPOLYGON (((-115.34289 32.58415, -115.3428..."
2,BAJA CALIFORNIA,TIJUANA,TIJUANA,020040001,02,004,0001,2062618.0,Prioridad 1: Ampliamente urbana,None,None,localidad,020040001,NaN,"MULTIPOLYGON (((-116.95397 32.39583, -116.9539..."
3,BAJA CALIFORNIA,PLAYAS DE ROSARITO,PLAYAS DE ROSARITO,020050001,02,005,0001,123240.0,Prioridad 1: Ampliamente urbana,None,None,localidad,020050001,NaN,"MULTIPOLYGON (((-117.0777 32.39365, -117.07772..."
4,BAJA CALIFORNIA SUR,LA PAZ,LA PAZ,030030001,03,003,0001,278809.0,Prioridad 1: Ampliamente urbana,None,None,localidad,030030001,NaN,"MULTIPOLYGON (((-110.31242 24.022, -110.31232 ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3605,Zacatecas,Ojocaliente,Ojocaliente,320360001,32,32036,0001,NaN,Prioridad 2: Segunda prioridad,None,None,localidad,320360001,NaN,"MULTIPOLYGON (((-102.26118 22.58854, -102.2609..."
3606,Zacatecas,Tlaltenango de Sánchez Román,Tlaltenango de Sánchez Román,320480001,32,32048,0001,NaN,Prioridad 2: Segunda prioridad,None,None,localidad,320480001,NaN,"MULTIPOLYGON (((-103.29447 21.79267, -103.2945..."
3607,Zacatecas,Nochistlán de Mejía,Nochistlán de Mejía,320340001,32,32034,0001,NaN,Prioridad 2: Segunda prioridad,None,None,localidad,320340001,NaN,"MULTIPOLYGON (((-102.8483 21.38342, -102.84837..."
3608,Zacatecas,Jalpa,Jalpa,320190001,32,32019,0001,NaN,Prioridad 2: Segunda prioridad,None,None,localidad,320190001,NaN,"MULTIPOLYGON (((-102.96931 21.65781, -102.9694..."


In [28]:
# base = base[base['tipo_mge'] == 'ageb']

In [29]:
conteo_consul = base [ 'consultorios_faltantes'].unique()
conteo_consul

array([nan, 0.5, 0.2, 0.7, 0.1, 0. , 0.4, 1.9, 2.4, 1.8, 0.8, 0.3, 0.6,
       0.9, 1.1, 2. , 1. , 1.3, 1.4, 2.2, 2.3, 4.3, 2.8, 1.5, 1.2, 1.6,
       2.1, 5.4, 2.7, 2.6, 6.7, 3.5, 3. , 1.7, 2.9, 3.3, 2.5, 3.2, 3.1,
       4.2, 3.4, 3.8, 4.7, 7.8])

In [30]:
base.columns

Index(['nom_ent', 'nom_mun', 'nom_loc', 'cve_loc', 'entidad', 'mun', 'loc',
       'poblacion_total_2026', 'tipo', 'cve_ageb', 'observacion', 'tipo_mge',
       'cvegeo', 'consultorios_faltantes', 'geometry'],
      dtype='object')

In [35]:


columnas_requeridas = {"cve_loc", "nom_loc", "consultorios_faltantes", "poblacion_total_2026", "geometry"}
columnas_faltantes = columnas_requeridas.difference(base.columns)
if columnas_faltantes:
    raise ValueError(
        f"Faltan columnas requeridas en la base: {sorted(columnas_faltantes)}"
    )

base = base[["cve_loc", "nom_loc", "consultorios_faltantes", "poblacion_total_2026", "geometry"]].copy()
base["cve_loc"] = base["cve_loc"].astype("string").str.strip()
base["nom_loc"] = base["nom_loc"].astype("string").str.strip()
base["consultorios_faltantes"] = base["consultorios_faltantes"].astype("Float64")
base["poblacion_total_2026"] = base["poblacion_total_2026"].astype("Float64")
base = base[
    base["consultorios_faltantes"].isna()
    | (base["consultorios_faltantes"] != 0)
]
base = base.dropna(subset=["cve_loc", "nom_loc", "geometry"])
base = base[base.geometry.is_valid & ~base.geometry.is_empty]
base = base.drop_duplicates(subset=["cve_loc"], keep="first")

if base.crs is None:
    raise ValueError("La base no tiene CRS definido; no es seguro transformar geometry.")

base = base.to_crs("EPSG:6372")
base["geometry"] = base.geometry.simplify(100, preserve_topology=True)
base = base.to_crs("EPSG:4326")
RUTA_SALIDA.parent.mkdir(parents=True, exist_ok=True)
base.to_file(RUTA_SALIDA, driver="GeoJSON")

print(f"Registros exportados: {len(base):,}")
print(f"CRS de salida: {base.crs}")
print(f"Archivo generado: {RUTA_SALIDA}")

Registros exportados: 2,298
CRS de salida: EPSG:4326
Archivo generado: c:\Users\jose.valdez\Downloads\nuevo-map\mapa-AGEB-\public\mapa_base.geojson
